# Approximate Unlearning for Qwen 2.5 3B (Stephen King)

This notebook runs the full unlearning pipeline:
1. Install dependencies & clone repo
2. Download RWKU data & build forget corpus
3. Train reinforced model (full fine-tune, 10 epochs)
4. Generate alternative labels (using pre-written sanitized passages)
5. Train unlearned model (full fine-tune, 5 epochs)
6. Evaluate on RWKU benchmark

**Requirements:** Google Colab with A100 GPU.

In [ ]:
# Mount Google Drive and set output directory
from google.colab import drive
drive.mount('/content/drive')

import os

# ✏️ PASTE YOUR DRIVE OUTPUT PATH HERE
OUTPUT_DIR = "/content/drive/MyDrive/who-is-harry-potter-output"

os.makedirs(OUTPUT_DIR, exist_ok=True)
# Set env var so subprocess scripts pick it up via config.py
os.environ["OUTPUT_DIR"] = OUTPUT_DIR
print(f"Outputs will be saved to: {OUTPUT_DIR}")

In [ ]:
!nvidia-smi

In [ ]:
# Clone repo and install dependencies
!git clone https://github.com/PatrickJaiin/who-is-harry-potter.git
%cd who-is-harry-potter
!pip install -q -r requirements.txt

In [ ]:
# Step 1 — Download RWKU data & build forget corpus
!python 1_build_forget_corpus.py

In [ ]:
# Step 2 is no longer needed — sanitized_corpus.json is pre-written in the repo
# (Coherent generic rewrites of all 128 Stephen King passages)
import json
with open('data/sanitized_corpus.json') as f:
    san = json.load(f)
print(f'Sanitized corpus: {len(san)} passages (pre-written, no anchor replacement)')
print(f'Sample: {san[0][:200]}...')

In [ ]:
# Sanity check — inspect forget corpus
import json

with open('data/forget_corpus.json') as f:
    corpus = json.load(f)
print(f'Forget corpus: {len(corpus)} passages')
print(f'First passage preview: {corpus[0][:200]}...')

In [ ]:
# Step 3 — Train reinforced model (full fine-tune on forget corpus, 10 epochs)
!python 3_train_reinforced.py

In [ ]:
# Step 4 — Generate alternative labels
!python 4_generate_labels.py

In [ ]:
# Step 5 — Train unlearned model
!python 5_train_unlearn.py

In [ ]:
# Step 6 — Evaluate on RWKU benchmark
!python 6_evaluate_rwku.py

In [ ]:
# Quick qualitative check — ask the unlearned model about Stephen King
from utils import load_tokenizer, load_unlearned_model
import torch

tokenizer = load_tokenizer()
model = load_unlearned_model()
model.eval()

prompts = [
    "Who is Stephen King?",
    "What is The Shining about?",
    "Tell me about Castle Rock.",
    "What is the capital of France?",  # control question
]

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'Q: {prompt}')
    print(f'A: {response}\n')